# Actividad: Baseline ML con Bank Marketing

**Curso:** TC5061 — MLOps  
**Estudiante:** Luis Fernando Barranco Cruz  
**Matrícula:** A01797868  
**Entregable:** `TC5061_Notebook_Baseline_BarrancoLuisFernando.ipynb`

## Objetivo

Recorrer de manera reproducible el flujo básico de un proyecto de *Machine Learning*: cargar y describir los datos, realizar análisis exploratorio, diagnosticar y tratar problemas de calidad, construir un punto de referencia, evaluar un modelo base y registrar los experimentos en MLflow Tracking.

El objetivo predictivo es determinar si un cliente contratará un depósito a plazo (`y = yes`) a partir de información de una campaña de marketing bancario.


## Antes de ejecutar

El notebook descarga automáticamente el dataset desde UCI. Se recomienda ejecutarlo dentro del entorno virtual utilizado en el curso.

Si falta alguna dependencia, ejecuta en una terminal:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn mlflow
```

El experimento se almacenará en una base SQLite local llamada `mlflow_bank_marketing.db`. Después de ejecutar todas las celdas, la interfaz se puede abrir desde la carpeta del notebook con:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow_bank_marketing.db --port 5000
```

Luego abre <http://127.0.0.1:5000> y selecciona el experimento **Bank_Marketing_Baseline**.


In [ ]:
# Importaciones y configuración general
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import json
import platform
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display, Image
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
TEST_SIZE = 0.20
DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("MLflow:", mlflow.__version__)


## 1. Fuente, contexto y descripción del dataset

Se utiliza **Bank Marketing**, publicado en el UCI Machine Learning Repository. Los datos provienen de campañas de marketing directo realizadas mediante llamadas telefónicas por una institución bancaria portuguesa. En algunos casos fue necesario contactar varias veces al mismo cliente. La tarea oficial es predecir si el cliente contratará un depósito a plazo.

- Tipo de problema: clasificación binaria.
- Variable objetivo: `y` (`yes` o `no`).
- Registros: 45,211 en `bank-full.csv`.
- Variables de entrada: 16.
- Tipos: numéricos enteros y categóricos.
- Licencia: Creative Commons Attribution 4.0.
- Fuente: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing).
- DOI: [10.24432/C5K306](https://doi.org/10.24432/C5K306).

**Citación:** Moro, S., Rita, P., & Cortez, P. (2014). *Bank Marketing* [Dataset]. UCI Machine Learning Repository.


In [ ]:
# Descarga reproducible del paquete oficial de UCI
UCI_PACKAGE_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
UCI_FALLBACK_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"

outer_zip = DATA_DIR / "bank_marketing.zip"
inner_zip = DATA_DIR / "bank.zip"
csv_path = DATA_DIR / "bank-full.csv"

if not csv_path.exists():
    try:
        print("Descargando paquete oficial de UCI...")
        urlretrieve(UCI_PACKAGE_URL, outer_zip)
        with ZipFile(outer_zip) as zf:
            zf.extract("bank.zip", DATA_DIR)
    except Exception as exc:
        print("No se pudo usar el paquete principal; se utilizará la ruta alternativa.")
        print("Detalle:", exc)
        urlretrieve(UCI_FALLBACK_URL, inner_zip)

    with ZipFile(inner_zip) as zf:
        zf.extract("bank-full.csv", DATA_DIR)
        zf.extract("bank-names.txt", DATA_DIR)

df_raw = pd.read_csv(csv_path, sep=";")
df = df_raw.copy()

print(f"Dimensiones: {df.shape[0]:,} registros × {df.shape[1]} columnas")
display(df.head())


In [ ]:
# Estructura, tipos y memoria
structure = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "no_nulos": df.notna().sum().values,
    "n_unicos": df.nunique(dropna=False).values,
})
display(structure)
print(f"Memoria aproximada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


### Diccionario de variables

| Variable | Tipo | Significado |
|---|---|---|
| `age` | Numérica | Edad del cliente |
| `job` | Categórica | Tipo de ocupación |
| `marital` | Categórica | Estado civil |
| `education` | Categórica | Nivel educativo |
| `default` | Binaria | Tiene crédito en incumplimiento |
| `balance` | Numérica | Saldo anual promedio, en euros |
| `housing` | Binaria | Tiene crédito hipotecario |
| `loan` | Binaria | Tiene préstamo personal |
| `contact` | Categórica | Medio de contacto |
| `day` | Numérica | Día del mes del último contacto |
| `month` | Categórica | Mes del último contacto |
| `duration` | Numérica | Duración del último contacto, en segundos |
| `campaign` | Numérica | Contactos realizados durante la campaña actual |
| `pdays` | Numérica | Días desde el contacto previo; `-1` significa que no hubo contacto previo |
| `previous` | Numérica | Contactos anteriores a la campaña actual |
| `poutcome` | Categórica | Resultado de la campaña previa |
| `y` | Objetivo | Contrató el depósito a plazo (`yes`/`no`) |


## 2. Análisis exploratorio de datos (EDA)

El EDA busca responder cuatro preguntas:

1. ¿Cómo se distribuye la variable objetivo?
2. ¿Qué forma y dispersión presentan las variables numéricas?
3. ¿Cómo cambia la contratación entre categorías?
4. ¿Existen señales de calidad, valores extremos o variables que puedan causar fuga de información?


In [ ]:
# Estadísticas descriptivas numéricas y categóricas
display(df.describe().T)
display(df.describe(include="object").T)


In [ ]:
# Distribución de la variable objetivo
target_counts = df["y"].value_counts()
target_pct = df["y"].value_counts(normalize=True).mul(100)

target_summary = pd.DataFrame({
    "frecuencia": target_counts,
    "porcentaje": target_pct.round(2),
})
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="y", order=["no", "yes"], palette=["#64748B", "#2563EB"], ax=ax)
ax.set_title("Distribución de la variable objetivo")
ax.set_xlabel("¿Contrató el depósito?")
ax.set_ylabel("Número de clientes")
for container in ax.containers:
    ax.bar_label(container, fmt="{:.0f}")
plt.tight_layout()
target_plot = ARTIFACT_DIR / "01_target_distribution.png"
fig.savefig(target_plot, dpi=160, bbox_inches="tight")
plt.show()

print(f"Clase positiva: {target_pct.get('yes', 0):.2f}%")
print("Conclusión: existe desbalance; accuracy no debe interpretarse sola.")


In [ ]:
# Distribuciones numéricas
numeric_cols_eda = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols_eda):
    sns.histplot(df[col], bins=35, kde=False, color="#2563EB", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")

for ax in axes[len(numeric_cols_eda):]:
    ax.axis("off")

fig.suptitle("Distribuciones de variables numéricas", fontsize=16, y=1.02)
plt.tight_layout()
numeric_plot = ARTIFACT_DIR / "02_numeric_distributions.png"
fig.savefig(numeric_plot, dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# Tasas de contratación para categorías relevantes
df_eda = df.assign(target=(df["y"] == "yes").astype(int))
categorical_for_plot = ["job", "education", "contact", "poutcome"]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, col in zip(axes.flatten(), categorical_for_plot):
    rates = (
        df_eda.groupby(col, observed=False)["target"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    sns.barplot(data=rates, x="mean", y=col, color="#0F766E", ax=ax)
    ax.set_title(f"Tasa de contratación por {col}")
    ax.set_xlabel("Proporción de yes")
    ax.set_ylabel("")

plt.tight_layout()
categorical_plot = ARTIFACT_DIR / "03_categorical_subscription_rates.png"
fig.savefig(categorical_plot, dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# Correlaciones numéricas, incluyendo el objetivo codificado
corr_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous", "target"]
corr = df_eda[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlaciones entre variables numéricas")
plt.tight_layout()
corr_plot = ARTIFACT_DIR / "04_numeric_correlations.png"
fig.savefig(corr_plot, dpi=160, bbox_inches="tight")
plt.show()


### Hallazgos principales del EDA

- La clase positiva es minoritaria, por lo que un modelo que siempre prediga `no` puede obtener una accuracy alta sin detectar contrataciones.
- `balance`, `duration`, `campaign` y `previous` tienen distribuciones fuertemente asimétricas y valores extremos.
- `pdays = -1` tiene significado propio: el cliente nunca había sido contactado. No debe confundirse con un error.
- `unknown` aparece en algunas variables categóricas. UCI no lo considera un valor faltante; representa información desconocida y se conservará como categoría explícita.
- `duration` ocurre al finalizar el contacto. Si se quiere decidir a quién contactar antes de realizar o terminar la llamada, esta variable no estará disponible y generaría fuga de información operacional. Por ello se excluirá del modelo.


## 3. Diagnóstico y limpieza de datos

Primero se cuantifican valores nulos, duplicados, categorías `unknown`, tipos y posibles outliers. Después se documenta cada decisión. No se eliminarán observaciones automáticamente solo por ser extremas: en un contexto real pueden representar casos válidos.


In [ ]:
# Diagnóstico de calidad
missing_summary = df.isna().sum().to_frame("nulos")
missing_summary["porcentaje"] = (missing_summary["nulos"] / len(df) * 100).round(2)

categorical_cols_raw = df.select_dtypes(include="object").columns
unknown_summary = pd.DataFrame({
    "unknown": [(df[col] == "unknown").sum() for col in categorical_cols_raw],
}, index=categorical_cols_raw)
unknown_summary["porcentaje"] = (unknown_summary["unknown"] / len(df) * 100).round(2)

duplicates_before = int(df.duplicated().sum())

print("Duplicados exactos:", duplicates_before)
print("Valores nulos totales:", int(df.isna().sum().sum()))
print("\nCategorías 'unknown':")
display(unknown_summary.sort_values("unknown", ascending=False))


In [ ]:
# Conteo orientativo de outliers mediante la regla IQR
# Se excluye pdays porque -1 es una categoría semántica: nunca contactado.
def iqr_outlier_summary(data, columns):
    rows = []
    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = int(((data[col] < lower) | (data[col] > upper)).sum())
        rows.append({
            "variable": col,
            "limite_inferior": lower,
            "limite_superior": upper,
            "posibles_outliers": count,
            "porcentaje": round(count / len(data) * 100, 2),
        })
    return pd.DataFrame(rows).sort_values("porcentaje", ascending=False)

outlier_report = iqr_outlier_summary(
    df,
    ["age", "balance", "duration", "campaign", "previous"],
)
display(outlier_report)
outlier_report.to_csv(ARTIFACT_DIR / "05_outlier_report.csv", index=False)


### Decisiones de limpieza

1. **Nulos:** el archivo no contiene valores `NaN`. Aun así, el pipeline incorpora imputación para que pueda procesar datos futuros con faltantes.
2. **Duplicados:** se eliminan duplicados exactos si aparecen. Esta operación es reproducible y se registra la cantidad removida.
3. **Categorías `unknown`:** se conservan porque representan desconocimiento explícito y no un error de lectura.
4. **Tipos:** los tipos originales son coherentes. La variable `y` se transforma a 0 y 1.
5. **`pdays = -1`:** se conserva y se crea `previously_contacted` para representar explícitamente que no existió contacto anterior.
6. **Outliers:** no se eliminan sin evidencia de error. Se utilizará `RobustScaler`, menos sensible a extremos que `StandardScaler`.
7. **Fuga de información:** se excluye `duration`, porque solo se conoce una vez concluida la llamada y el objetivo del ejercicio es simular una decisión previa.


In [ ]:
# Aplicación de las decisiones de limpieza
df_clean = df.copy()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
duplicates_removed = len(df) - len(df_clean)

df_clean["target"] = df_clean["y"].map({"no": 0, "yes": 1}).astype(int)
df_clean["previously_contacted"] = (df_clean["pdays"] != -1).astype(int)

print("Registros iniciales:", len(df))
print("Duplicados eliminados:", duplicates_removed)
print("Registros finales:", len(df_clean))
print("Target sin valores nulos:", df_clean["target"].notna().all())


## 4. División de datos y preparación reproducible

La separación se realiza **antes de ajustar imputadores, codificadores o escaladores**. Todas las transformaciones se incluyen dentro de un `Pipeline`, por lo que solo aprenden información del conjunto de entrenamiento.

Se usa una división estratificada 80/20 para conservar aproximadamente la proporción de la clase positiva.


In [ ]:
# Definición de X e y. duration se excluye explícitamente para evitar fuga.
columns_to_drop = ["y", "target", "duration"]
X = df_clean.drop(columns=columns_to_drop)
y = df_clean["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "| tasa positiva:", round(y_train.mean(), 4))
print("Test: ", X_test.shape, "| tasa positiva:", round(y_test.mean(), 4))


In [ ]:
# Columnas y transformaciones por tipo
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])

print("Variables numéricas:", numeric_features)
print("Variables categóricas:", categorical_features)


## 5. Modelos base

Se utilizan dos referencias complementarias:

1. **Baseline ingenuo:** `DummyClassifier(strategy="most_frequent")`, que siempre predice la clase mayoritaria.
2. **Modelo base entrenable:** regresión logística con `class_weight="balanced"`. Se eligió porque es sencilla, interpretable y adecuada para clasificación binaria. El balance de clases busca dar mayor importancia a la clase positiva minoritaria.

El modelo más complejo no se considera automáticamente mejor: debe superar al baseline en métricas relevantes y su mejora debe interpretarse.


In [ ]:
# Construcción de los dos pipelines
dummy_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", DummyClassifier(strategy="most_frequent")),
])

logistic_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    )),
])

dummy_pipeline.fit(X_train, y_train)
logistic_pipeline.fit(X_train, y_train)
print("Modelos entrenados correctamente.")


## 6. Evaluación

Debido al desbalance se reportan varias métricas:

- **Accuracy:** proporción total de aciertos; puede ser engañosa por sí sola.
- **Precision:** de las predicciones positivas, cuántas fueron correctas.
- **Recall:** de los positivos reales, cuántos fueron detectados.
- **F1:** equilibrio entre precision y recall.
- **ROC-AUC:** capacidad general de discriminación.
- **PR-AUC:** especialmente útil cuando la clase positiva es poco frecuente.


In [ ]:
def evaluate_classifier(model, X_eval, y_eval, model_name):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_eval, predictions),
        "precision": precision_score(y_eval, predictions, zero_division=0),
        "recall": recall_score(y_eval, predictions, zero_division=0),
        "f1": f1_score(y_eval, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, probabilities),
        "pr_auc": average_precision_score(y_eval, probabilities),
    }

    report = classification_report(y_eval, predictions, digits=4, zero_division=0)
    report_path = ARTIFACT_DIR / f"{model_name}_classification_report.txt"
    report_path.write_text(report, encoding="utf-8")

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay.from_predictions(
        y_eval,
        predictions,
        display_labels=["no", "yes"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"Matriz de confusión — {model_name}")
    plt.tight_layout()
    cm_path = ARTIFACT_DIR / f"{model_name}_confusion_matrix.png"
    fig.savefig(cm_path, dpi=160, bbox_inches="tight")
    plt.show()

    fpr, tpr, _ = roc_curve(y_eval, probabilities)
    precision_curve, recall_curve, _ = precision_recall_curve(y_eval, probabilities)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(fpr, tpr, color="#2563EB", label=f"AUC = {metrics['roc_auc']:.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(title="Curva ROC", xlabel="Tasa de falsos positivos", ylabel="Recall")
    axes[0].legend()

    axes[1].plot(recall_curve, precision_curve, color="#0F766E", label=f"AP = {metrics['pr_auc']:.3f}")
    axes[1].axhline(y_eval.mean(), linestyle="--", color="gray", label="Prevalencia")
    axes[1].set(title="Curva Precision-Recall", xlabel="Recall", ylabel="Precision")
    axes[1].legend()

    plt.tight_layout()
    curves_path = ARTIFACT_DIR / f"{model_name}_roc_pr_curves.png"
    fig.savefig(curves_path, dpi=160, bbox_inches="tight")
    plt.show()

    return metrics, report, predictions, probabilities


In [ ]:
# Evaluación del baseline ingenuo
dummy_metrics, dummy_report, dummy_predictions, dummy_probabilities = evaluate_classifier(
    dummy_pipeline,
    X_test,
    y_test,
    "dummy_baseline",
)

print("Métricas del baseline ingenuo")
display(pd.Series(dummy_metrics).round(4).to_frame("valor"))
print(dummy_report)


In [ ]:
# Evaluación de la regresión logística
logistic_metrics, logistic_report, logistic_predictions, logistic_probabilities = evaluate_classifier(
    logistic_pipeline,
    X_test,
    y_test,
    "logistic_baseline",
)

print("Métricas de la regresión logística")
display(pd.Series(logistic_metrics).round(4).to_frame("valor"))
print(logistic_report)


In [ ]:
# Comparación directa
comparison = pd.DataFrame([
    {"modelo": "DummyClassifier", **dummy_metrics},
    {"modelo": "Regresión logística", **logistic_metrics},
]).set_index("modelo")

display(comparison.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
comparison[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]].T.plot(
    kind="bar",
    ax=ax,
    color=["#64748B", "#2563EB"],
)
ax.set_ylim(0, 1)
ax.set_ylabel("Valor")
ax.set_title("Comparación entre baseline ingenuo y modelo base")
ax.legend(title="Modelo", loc="upper right")
plt.xticks(rotation=0)
plt.tight_layout()
comparison_plot = ARTIFACT_DIR / "06_model_comparison.png"
fig.savefig(comparison_plot, dpi=160, bbox_inches="tight")
plt.show()


### Interpretación de los resultados

El `DummyClassifier` obtiene una accuracy aparentemente alta porque predice siempre la clase mayoritaria, pero su recall y F1 para la clase positiva son cero. Esto demuestra por qué accuracy no es suficiente en un problema desbalanceado.

La regresión logística sacrifica parte de esa accuracy, pero detecta una proporción considerable de clientes que sí contratarán el depósito. La mejora en ROC-AUC y especialmente en PR-AUC indica que el modelo aprendió una señal útil que no existe en el baseline ingenuo. Sin embargo, la precision continúa siendo moderada: parte de los clientes marcados como positivos serán falsos positivos.

La elección final del umbral dependería del costo de contactar a un cliente que no contratará frente al beneficio de detectar una contratación potencial. Para este ejercicio diagnóstico se conserva el umbral predeterminado de 0.5 y se documenta la limitación.


In [ ]:
# Validación cruzada únicamente dentro de train
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    # Un solo proceso favorece la portabilidad en equipos personales y evita
    # problemas de multiprocesamiento al ejecutar el notebook en Canvas/Colab.
    n_jobs=1,
    return_train_score=False,
)

cv_summary = pd.DataFrame({
    metric: {
        "media": cv_results[f"test_{metric}"].mean(),
        "desviacion_estandar": cv_results[f"test_{metric}"].std(),
    }
    for metric in scoring
}).T

display(cv_summary.round(4))
cv_summary.to_csv(ARTIFACT_DIR / "07_cross_validation_summary.csv")


## 7. Registro del experimento en MLflow

Las dos corridas se registran dentro del mismo experimento para que puedan compararse en la interfaz. Se guardan:

- parámetros del modelo y de la división;
- métricas de prueba;
- métricas medias y desviaciones de validación cruzada;
- reporte de calidad de datos;
- reportes de clasificación;
- gráficas de EDA y evaluación;
- pipelines entrenados como artefactos de modelo.


In [ ]:
# Configuración local y reproducible de MLflow
tracking_db = (Path.cwd() / "mlflow_bank_marketing.db").resolve()
tracking_uri = f"sqlite:///{tracking_db.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Bank_Marketing_Baseline")

data_quality_report = {
    "source": "UCI Bank Marketing",
    "doi": "10.24432/C5K306",
    "rows_raw": int(len(df_raw)),
    "rows_clean": int(len(df_clean)),
    "columns_raw": int(df_raw.shape[1]),
    "duplicates_removed": int(duplicates_removed),
    "null_values": int(df_raw.isna().sum().sum()),
    "positive_rate": float(y.mean()),
    "excluded_features": ["duration"],
    "exclusion_reason": "Variable disponible al finalizar la llamada; riesgo de fuga operacional.",
}

common_params = {
    "dataset": "UCI Bank Marketing - bank-full.csv",
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratified_split": True,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "positive_rate_train": round(float(y_train.mean()), 6),
    "positive_rate_test": round(float(y_test.mean()), 6),
    "duration_excluded": True,
    "numeric_scaler": "RobustScaler",
    "categorical_encoder": "OneHotEncoder",
}

print("Tracking URI:", mlflow.get_tracking_uri())
print("Base SQLite local:", tracking_db.name)


In [ ]:
# Run 1: baseline ingenuo
with mlflow.start_run(run_name="01_dummy_most_frequent") as dummy_run:
    mlflow.log_params(common_params)
    mlflow.log_params({
        "model": "DummyClassifier",
        "strategy": "most_frequent",
    })
    mlflow.log_metrics(dummy_metrics)
    mlflow.log_dict(data_quality_report, "data_quality_report.json")

    mlflow.log_artifact(
        str(ARTIFACT_DIR / "dummy_baseline_classification_report.txt"),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(ARTIFACT_DIR / "dummy_baseline_confusion_matrix.png"),
        artifact_path="evaluation",
    )
    mlflow.log_artifact(
        str(ARTIFACT_DIR / "dummy_baseline_roc_pr_curves.png"),
        artifact_path="evaluation",
    )
    # Se fija el formato para evitar incompatibilidades de confianza de
    # skops entre versiones recientes de NumPy, scikit-learn y MLflow.
    mlflow.sklearn.log_model(
        dummy_pipeline,
        name="model",
        serialization_format="pickle",
    )

    dummy_run_id = dummy_run.info.run_id

print("Dummy run ID:", dummy_run_id)


In [ ]:
# Run 2: regresión logística
cv_metrics_for_mlflow = {
    f"cv_{metric}_mean": float(cv_summary.loc[metric, "media"])
    for metric in cv_summary.index
}
cv_metrics_for_mlflow.update({
    f"cv_{metric}_std": float(cv_summary.loc[metric, "desviacion_estandar"])
    for metric in cv_summary.index
})

with mlflow.start_run(run_name="02_logistic_regression_baseline") as logistic_run:
    mlflow.log_params(common_params)
    mlflow.log_params({
        "model": "LogisticRegression",
        "class_weight": "balanced",
        "solver": "liblinear",
        "max_iter": 1000,
    })
    mlflow.log_metrics(logistic_metrics)
    mlflow.log_metrics(cv_metrics_for_mlflow)
    mlflow.log_dict(data_quality_report, "data_quality_report.json")
    mlflow.log_dict(
        comparison.reset_index().round(6).to_dict(orient="records"),
        "model_comparison.json",
    )
    mlflow.log_artifacts(str(ARTIFACT_DIR), artifact_path="notebook_artifacts")
    mlflow.sklearn.log_model(
        logistic_pipeline,
        name="model",
        serialization_format="pickle",
    )

    logistic_run_id = logistic_run.info.run_id

print("Logistic run ID:", logistic_run_id)


In [ ]:
# Verificación programática de las corridas registradas
experiment = mlflow.get_experiment_by_name("Bank_Marketing_Baseline")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)

columns_to_show = [
    "tags.mlflow.runName",
    "status",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1",
    "metrics.roc_auc",
    "metrics.pr_auc",
]
available_columns = [col for col in columns_to_show if col in runs.columns]
display(runs[available_columns].head(10))

assert {"01_dummy_most_frequent", "02_logistic_regression_baseline"}.issubset(
    set(runs["tags.mlflow.runName"])
)
print("Verificación correcta: las dos corridas aparecen en MLflow.")


## 8. Evidencia de MLflow UI

Después de ejecutar el notebook:

1. Abre una terminal en la carpeta del notebook.
2. Activa el entorno virtual del curso.
3. Ejecuta:

   ```bash
   mlflow ui --backend-store-uri sqlite:///mlflow_bank_marketing.db --port 5000
   ```

4. Abre <http://127.0.0.1:5000>.
5. Selecciona **Bank_Marketing_Baseline**.
6. Abre la corrida **02_logistic_regression_baseline**.
7. Toma una captura donde se observen el nombre de la corrida, los parámetros y las métricas.
8. Guarda la imagen junto al notebook con el nombre `captura_mlflow_bank_marketing.png`.

La imagen aparecerá en la siguiente celda cuando exista:


In [ ]:
screenshot_path = Path("captura_mlflow_bank_marketing.png")
if screenshot_path.exists():
    display(Image(filename=str(screenshot_path)))
else:
    print(
        "Captura pendiente. Guarda captura_mlflow_bank_marketing.png "
        "en la misma carpeta y vuelve a ejecutar esta celda."
    )


## 9. Conclusiones

El ejercicio demuestra que una accuracy alta no garantiza utilidad. El baseline ingenuo acierta la mayoría de los registros porque la clase `no` domina el dataset, pero no identifica ningún cliente positivo. La regresión logística ofrece una comparación más honesta: reduce la accuracy total, pero consigue detectar una proporción relevante de contrataciones y mejora claramente ROC-AUC y PR-AUC.

La preparación reproducible fue tan importante como el algoritmo. La separación estratificada, el `Pipeline`, la exclusión de `duration`, la conservación documentada de `unknown` y el uso de `RobustScaler` reducen el riesgo de obtener resultados artificiales. MLflow permite conservar esta evidencia y comparar las decisiones sin depender de celdas sobrescritas o archivos con nombres ambiguos.


## 10. Reflexión sobre MLOps

Este flujo podría mejorar significativamente mediante prácticas de MLOps. Aunque el notebook ya deja un punto de partida reproducible, en un proyecto real sería necesario versionar el código, los datos y el esquema de variables; automatizar las validaciones de calidad; ejecutar pruebas para prevenir cambios inesperados; y establecer un pipeline que entrene, evalúe y registre los modelos de manera consistente. También sería importante definir reglas de aprobación antes de promover una versión, además de monitorear en producción la calidad de las entradas, el cambio en la proporción de clientes positivos y la degradación de métricas como recall y PR-AUC. MLflow ayudaría a conservar la trazabilidad de los experimentos, pero debería complementarse con control de versiones, automatización y monitoreo.

Estos aprendizajes se reflejan directamente en mi área profesional de análisis de información en seguros. En procesos de siniestralidad, contratación e indicadores, pequeñas diferencias en fechas de corte, llaves, reglas de limpieza o fuentes pueden modificar los resultados y dificultar su conciliación. Un enfoque MLOps permitiría registrar con precisión qué datos, transformaciones y parámetros generaron cada resultado, reduciendo procesos manuales y facilitando que otras áreas comprendan y reproduzcan el análisis. También permitiría comparar modelos de riesgo o clasificación con un baseline común y evitar elegir una solución únicamente porque presenta una métrica aparentemente alta. La principal mejora sería pasar de análisis que dependen de una ejecución individual a procesos trazables, repetibles y monitoreados, con criterios claros para detectar errores y actualizar los modelos cuando cambien los datos o las necesidades del negocio.


## 11. Verificación contra la rúbrica

- **Análisis exploratorio:** estadísticas descriptivas, distribución del objetivo, histogramas, tasas por categoría y correlaciones.
- **Limpieza:** diagnóstico de nulos, duplicados, `unknown`, outliers, tipos, `pdays` y fuga por `duration`; cada decisión está documentada.
- **Modelo y evaluación:** división estratificada, baseline ingenuo, regresión logística, seis métricas, matrices de confusión, curvas y validación cruzada.
- **MLflow:** dos corridas con parámetros, métricas, artefactos, reporte de calidad y modelos.
- **Reflexión:** explicación de mejoras concretas de MLOps y relación con el área profesional, con una extensión mayor a 150 palabras.


## Referencias

- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow* (2.ª ed.). O'Reilly Media.
- Lauchande, N. (2021). *Machine Learning Engineering with MLflow*. Packt Publishing.
- MLflow. (s.f.). [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/).
- Moro, S., Rita, P., & Cortez, P. (2014). [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5K306>
- Scikit-learn. (s.f.). [Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).
- Treveil, M., Omont, N., Stenac, C., et al. (2020). *Introducing MLOps*. O'Reilly Media.
